In [1]:
import pandas as pd
import numpy as np

In [2]:
# ============================================================
# 1. Load the dataset
# ============================================================

df = pd.read_csv(r"C:\Users\Kiruthiha\Documents\Kiruthiha_Docs\Virtual_Internships\JP_Morgan_Case_Quantitative_Researcher\Task 3 and 4_Loan_Data.csv")

# We only need FICO score and default status for this task
df = df[["fico_score", "default"]]

print("Dataset shape:", df.shape)

Dataset shape: (10000, 2)


In [3]:
# ============================================================
# 2. Aggregate the data by FICO score
# ============================================================

agg = (
    df.groupby("fico_score")
      .agg(
          total=("default", "count"),
          defaults=("default", "sum")
      )
      .reset_index()
)

agg = agg.sort_values("fico_score").reset_index(drop=True)

scores = agg["fico_score"].values
totals = agg["total"].values
defaults = agg["defaults"].values

# Number of unique FICO scores
n = len(scores)

print("Number of unique FICO scores:", n)

Number of unique FICO scores: 374


In [4]:
# ============================================================
# 3. Create cumulative totals
# ============================================================

cum_total = np.concatenate(
    ([0], np.cumsum(totals))
)

cum_default = np.concatenate(
    ([0], np.cumsum(defaults))
)

In [5]:
# ============================================================
# 4. Define the log-likelihood function
# ============================================================

def bucket_loglikelihood(i, j):

    # Total borrowers in the bucket
    N = cum_total[j + 1] - cum_total[i]

    # Total defaults in the bucket
    K = cum_default[j + 1] - cum_default[i]

    # Probability of default in this bucket
    p = K / N

    # Avoid log(0)
    # When there are either no defaults or all borrowers
    # default, the log-likelihood contribution is treated as 0.
    if p == 0 or p == 1:
        return 0

    # Log-likelihood
    return (
        K * np.log(p)
        + (N - K) * np.log(1 - p)
    )

In [6]:
# ============================================================
# 5. Set the number of FICO buckets
# ============================================================

num_buckets = 10

In [7]:
# ============================================================
# 6. Create Dynamic Programming tables
# ============================================================

dp = np.full(
    (num_buckets + 1, n),
    -np.inf
)

parent = np.zeros(
    (num_buckets + 1, n),
    dtype=int
)

In [8]:
# ============================================================
# 7. Base case: one bucket
# ============================================================

for j in range(n):
    dp[1][j] = bucket_loglikelihood(0, j)

In [9]:
# ============================================================
# 8. Dynamic Programming
# ============================================================

for k in range(2, num_buckets + 1):

    for j in range(k - 1, n):

        best = -np.inf
        best_split = 0

        # Try every possible location for the previous split
        for m in range(k - 2, j):

            # Log-likelihood of the new/final bucket
            current_bucket = bucket_loglikelihood(
                m + 1,
                j
            )

            # Total log-likelihood:
            # previous optimal solution + current bucket
            candidate = (
                dp[k - 1][m]
                + current_bucket
            )

            # Keep the split with the highest likelihood
            if candidate > best:
                best = candidate
                best_split = m

        # Store the best result
        dp[k][j] = best
        parent[k][j] = best_split

In [10]:
# ============================================================
# 9. Recover the optimal FICO bucket boundaries
# ============================================================

boundaries = []

# Start at the last FICO score
idx = n - 1

for k in range(num_buckets, 1, -1):

    split = parent[k][idx]

    # Store the actual FICO score at the split
    boundaries.append(scores[split])

    # Move to the previous bucket
    idx = split

# We recovered the boundaries backwards,
# so reverse them into ascending order.
boundaries.reverse()

print("\nOptimal FICO Boundaries:")
print(boundaries)


Optimal FICO Boundaries:
[np.int64(520), np.int64(552), np.int64(580), np.int64(611), np.int64(649), np.int64(696), np.int64(732), np.int64(752), np.int64(753)]


In [11]:
# ============================================================
# 10. Create the FICO-to-Rating mapping
# ============================================================
#   Lower rating = better credit score
# Therefore:
#   Highest FICO → Rating 1
#   Lowest FICO  → Rating 10

def assign_rating(fico):

    # Check the FICO score against the optimal boundaries
    for i, boundary in enumerate(boundaries):

        if fico <= boundary:

            return num_buckets - i

    return 1

# Apply the rating function to every borrower
df["rating"] = df["fico_score"].apply(assign_rating)

In [12]:
# ============================================================
# 11. Calculate PD for each rating
# ============================================================

rating_summary = (
    df.groupby("rating")
      .agg(
          Customers=("default", "count"),
          Defaults=("default", "sum"),
          PD=("default", "mean")
      )
      .sort_index()
)

In [13]:
# ============================================================
# 12. Display the final rating summary
# ============================================================

print("\nRating Summary:")
print(rating_summary)



Rating Summary:
        Customers  Defaults        PD
rating                               
1             242         5  0.020661
2               8         3  0.375000
3             303         5  0.016502
4            1104        64  0.057971
5            2609       256  0.098122
6            2465       402  0.163083
7            1561       381  0.244074
8             911       307  0.336992
9             496       229  0.461694
10            301       199  0.661130


In [14]:
# ============================================================
# 13. Create a clear rating map
# ============================================================
# This shows the FICO boundary associated with each rating.

rating_map = []

for rating in range(1, num_buckets + 1):

    # Find all FICO scores assigned to this rating
    fico_values = df.loc[
        df["rating"] == rating,
        "fico_score"
    ]

    rating_map.append({
        "Rating": rating,
        "Minimum_FICO": fico_values.min(),
        "Maximum_FICO": fico_values.max()
    })


rating_map = pd.DataFrame(rating_map)

print("\nFICO Rating Map:")
print(rating_map)


FICO Rating Map:
   Rating  Minimum_FICO  Maximum_FICO
0       1           754           850
1       2           753           753
2       3           733           752
3       4           697           732
4       5           650           696
5       6           612           649
6       7           581           611
7       8           553           580
8       9           521           552
9      10           408           520
